# 针对超市销售数据进行数据分析

## 一 目标：
该连锁超市希望了解近一年的销售情况，为运营决策提供数据支持。运营总监关心三个问题：
1. 哪些商品最受欢迎？是否应该调整商品结构？
2. 不同门店的业绩差距有多大？是否需要资源倾斜？
3. 一天中什么时段客流最大？如何优化排班和促销？
本次分析将针对这三个问题展开。

## 二 数据清洗

### 2.1导入数据

In [1]:
import pandas as pd

# 导入数据源，parse_dates：将时间字符串转为日期时间格式
data2017=pd.read_csv("order-14.3.csv",parse_dates=["成交时间"],encoding='gbk')
data2018=pd.read_csv("order-14.1.csv",parse_dates=["成交时间"],encoding='gbk')
print(data2017.shape)  #返回数据的行数与列数
print(data2018.shape)
data2017.head()

(3478, 7)
(3744, 7)


,商品ID,类别ID,门店编号,单价,销量,成交时间,订单ID
0,30006206,915000003,CDNL,25.23,0.328,2017-01-03 09:56:00,20170103CDLG000210052759
1,30163281,914010000,CDNL,2.00,2.000,2017-01-03 09:56:00,20170103CDLG000210052759
2,30200518,922000000,CDNL,19.62,0.230,2017-01-03 09:56:00,20170103CDLG000210052759
3,29989105,922000000,CDNL,2.80,2.044,2017-01-03 09:56:00,20170103CDLG000210052759
4,30179558,915000100,CDNL,47.41,0.226,2017-01-03 09:56:00,20170103CDLG000210052759


In [2]:
data2017.dtypes

商品ID             int64
类别ID             int64
门店编号               str
单价             float64
销量             float64
成交时间    datetime64[us]
订单ID               str
dtype: object

### 2.2检查数据异常

此为2017，2018某超市销售数据，分析数据可能出现的问题：
商品ID与类别ID是否可以重复？不可以的话是否有重复？
单价还有销量是否有负值？
时间是否有超出异常的值？
是否有缺失值？缺失比例多少？
#### 对于文本格式来说：
同一列字段格式是否统一？单位是否一致？
#### 对于此类分析，我认为单个异常的值不是很影响

In [3]:
# 查看 data2017 每列的唯一值数量
data2017.nunique()

商品ID    1069
类别ID     368
门店编号       3
单价       569
销量      1002
成交时间     510
订单ID     918
dtype: int64

这就说明：类别与商品ID可以重复

In [4]:
# 检查所有列完全重复的行
duplicated_mask =data2017.duplicated(keep=False)
# 查看所有重复的行（包括第一次出现）
data2017[duplicated_mask].sort_values(['订单ID', '商品ID'])

,商品ID,类别ID,门店编号,单价,销量,成交时间,订单ID
23,30133582,914050000,CDNL,1.50,1.0,2017-01-03 10:00:00,20170103CDLG000210052763
24,30133582,914050000,CDNL,1.50,1.0,2017-01-03 10:00:00,20170103CDLG000210052763
25,30133582,914050000,CDNL,1.50,1.0,2017-01-03 10:00:00,20170103CDLG000210052763
64,30020693,930040104,CDNL,22.52,1.0,2017-01-03 10:21:00,20170103CDLG000210052780
65,30020693,930040104,CDNL,22.52,1.0,2017-01-03 10:21:00,20170103CDLG000210052780
...,...,...,...,...,...,...,...
3309,30031960,915030104,CDXL,4.40,20.0,2017-01-03 06:32:00,20170103CDLG000510025099
3310,30031960,915030104,CDXL,4.40,20.0,2017-01-03 06:32:00,20170103CDLG000510025099
3311,30031960,915030104,CDXL,4.40,20.0,2017-01-03 06:32:00,20170103CDLG000510025099
3452,29998004,931020303,CDXL,3.12,1.0,2017-01-03 09:40:00,20170103CDLG000510025139


观察发现同时重复的出现在了一起很多

同时检查单价与销量是否有负值？

In [5]:
# 任一项为负的行
data2017[(data2017['单价'] < 0) | (data2017['销量'] < 0)]

,商品ID,类别ID,门店编号,单价,销量,成交时间,订单ID
748,29989077,922000008,CDNL,13.59,-0.544,2017-01-03 16:31:00,20170103CDLG000210052957
749,29989099,922000008,CDNL,1.58,-0.886,2017-01-03 16:31:00,20170103CDLG000210052957
750,30010045,911000000,CDNL,5.52,-1.000,2017-01-03 16:31:00,20170103CDLG000210052957
751,30022232,960000000,CDNL,0.30,-1.000,2017-01-03 16:31:00,20170103CDLG000210052957
752,30063823,914050100,CDNL,1.20,-1.000,2017-01-03 16:31:00,20170103CDLG000210052957
753,30023192,922000000,CDNL,9.62,-0.150,2017-01-03 16:31:00,20170103CDLG000210052957
754,30070164,914000001,CDNL,11.52,-1.000,2017-01-03 16:31:00,20170103CDLG000210052957
755,30136855,912060303,CDNL,17.43,-0.758,2017-01-03 16:31:00,20170103CDLG000210052957
756,29989059,922000003,CDNL,1.98,-1.869,2017-01-03 16:31:00,20170103CDLG000210052957
757,29989071,922000002,CDNL,3.40,-0.820,2017-01-03 16:31:00,20170103CDLG000210052957


存在负值行!

检查时间规律

In [6]:
# 查看时间范围
print("最早时间：", data2017['成交时间'].min())
print("最晚时间：", data2017['成交时间'].max())
# 查看时间范围
print("最早时间：", data2018['成交时间'].min())
print("最晚时间：", data2018['成交时间'].max())

最早时间： 2017-01-03 06:06:00
最晚时间： 2017-01-03 21:35:00
最早时间： 2017-02-01 00:00:00
最晚时间： 2018-02-28 00:00:00


第一张表格是2017.1.3，第二张表格包含2017，2018

In [7]:
data2017.isnull().sum()

商品ID    0
类别ID    0
门店编号    0
单价      0
销量      0
成交时间    0
订单ID    0
dtype: int64

In [8]:
data2018.isnull().sum()

商品ID    266
类别ID    266
门店编号    266
单价      266
销量      266
成交时间    266
订单ID    266
dtype: int64

结论：无缺失

### 2.3数据清洗

对于数据清理，我们要：完全重复记录，单价与销量出现负值，2018表中的数据；
对于表格中出现的销量，我默认为特殊产品，可以购买非完整项。

In [9]:
# 删除完全重复的行，保留第一次出现
data2017.drop_duplicates(inplace=True)
data2018.drop_duplicates(inplace=True)

print("去重后 data2017 形状：", data2017.shape)
print("去重后 data2018 形状：", data2018.shape)

去重后 data2017 形状： (3245, 7)
去重后 data2018 形状： (3469, 7)


In [10]:
data2017 = data2017[(data2017['单价'] >= 0) & (data2017['销量'] >= 0)]
data2018 = data2018[(data2018['单价'] >= 0) & (data2018['销量'] >= 0)]

In [11]:
# 保存清洗后的数据
data2017.to_csv('order_14.3_cleaned.csv', index=False, encoding='utf-8-sig')
data2018.to_csv('order_14.1_cleaned.csv', index=False, encoding='utf-8-sig')

### 再次检查


In [12]:
# 查看清洗后的数据概览
print("data2017 形状：", data2017.shape)
print("data2018 形状：", data2018.shape)

# 再次确认缺失值、负值、重复项已处理
print("缺失值：\n", data2017.isnull().sum())
print("负值检查：\n", (data2017[['单价', '销量']] < 0).sum())

# 查看数据类型
print(data2017.dtypes)

data2017 形状： (3231, 7)
data2018 形状： (3453, 7)
缺失值：
 商品ID    0
类别ID    0
门店编号    0
单价      0
销量      0
成交时间    0
订单ID    0
dtype: int64
负值检查：
 单价    0
销量    0
dtype: int64
商品ID             int64
类别ID             int64
门店编号               str
单价             float64
销量             float64
成交时间    datetime64[us]
订单ID               str
dtype: object


# 三数据分析


#### 先观察一下这两份数据的时间粒度

In [13]:
# data2017 的唯一日期（保留日期部分）
unique_dates_2017 = data2017['成交时间'].dt.floor('D').unique()
print(f"data2017 唯一日期数：{len(unique_dates_2017)}")
print(unique_dates_2017)

# data2018 的唯一日期
unique_dates_2018 = data2018['成交时间'].dt.floor('D').unique()
print(f"data2018 唯一日期数：{len(unique_dates_2018)}")
print(unique_dates_2018)

data2017 唯一日期数：1
<DatetimeArray>
['2017-01-03 00:00:00']
Length: 1, dtype: datetime64[us]
data2018 唯一日期数：86
<DatetimeArray>
['2018-01-01 00:00:00', '2018-01-02 00:00:00', '2018-01-03 00:00:00',
 '2018-01-04 00:00:00', '2018-01-05 00:00:00', '2018-01-06 00:00:00',
 '2018-01-07 00:00:00', '2018-01-08 00:00:00', '2018-01-09 00:00:00',
 '2018-01-10 00:00:00', '2018-01-11 00:00:00', '2018-01-12 00:00:00',
 '2018-01-13 00:00:00', '2018-01-14 00:00:00', '2018-01-15 00:00:00',
 '2018-01-16 00:00:00', '2018-01-17 00:00:00', '2018-01-18 00:00:00',
 '2018-01-19 00:00:00', '2018-01-20 00:00:00', '2018-01-21 00:00:00',
 '2018-01-22 00:00:00', '2018-01-23 00:00:00', '2018-01-24 00:00:00',
 '2018-01-25 00:00:00', '2018-01-26 00:00:00', '2018-01-27 00:00:00',
 '2018-01-28 00:00:00', '2018-01-29 00:00:00', '2018-01-30 00:00:00',
 '2018-02-01 00:00:00', '2018-02-02 00:00:00', '2018-02-03 00:00:00',
 '2018-02-04 00:00:00', '2018-02-05 00:00:00', '2018-02-06 00:00:00',
 '2018-02-07 00:00:00', '2018-02-08 

结论：data2017集中在一天（同一纬度）
     data2018分散在2017年二月与2018年二月
     可以用来横向分析


### 3.1哪些商品比较受欢迎


#### 对比2017.1.3同时间粒度下的各种商品的成交量

In [14]:
# 按商品ID分组，计算总销量和平均单价
type_stats = data2017.groupby('商品ID').agg(
    总销量=('销量', 'sum'),
    平均单价=('单价', 'mean')
).reset_index()

# 按总销量降序排列
type_stats = type_stats.sort_values('总销量', ascending=False)

print("各商品总销量及平均单价（前5）：")
print(type_stats.head())

各商品总销量及平均单价（前5）：
         商品ID      总销量      平均单价
8    29989059  382.461  2.177422
18   29989072  102.876  0.780313
469  30022232  100.000  0.300000
57   29989157   72.453  5.599815
476  30023041   64.416  3.200000


是否应该调整商品结构？


In [21]:

# 按总销量升序排列，取前10（销量最低的10个）
lowest_10 = type_stats.sort_values('总销量', ascending=True).head(10)
print("销量倒数前十的商品ID及其平均单价：")
print(lowest_10)

销量倒数前十的商品ID及其平均单价：
          商品ID    总销量     平均单价
930   30179536  0.040   49.630
1067  30214826  0.073   55.205
926   30179516  0.110  120.530
679   30100004  0.120   59.620
933   30179544  0.124   49.625
522   30031938  0.128   16.800
936   30179548  0.130   41.930
160   30002260  0.167   25.370
154   30002226  0.181   33.320
931   30179537  0.190   72.260


至于是否需要调整商品结构，则需要加入单价进行综合考虑，我们注意到前五于后10的价格差别较大，因此不能轻易调整

### 3.2不同门店的业绩差距有多大？是否需要资源倾斜？

定义业绩指标：

总销售额=单价*销量总和

In [16]:
# 计算销售额（乘以1000，可根据需要调整）
data2017["销售额"] = data2017["销量"] * data2017["单价"]
# 按门店编号汇总总销售额
store_total = data2017.groupby("门店编号")["销售额"].sum().reset_index()
store_total = store_total.sort_values("销售额", ascending=False)

print(store_total)

   门店编号          销售额
0  CDLG  10493.36542
2  CDXL   9584.16710
1  CDNL   7561.27355


In [17]:
total = data2017["销售额"].sum()
store_total["占比"] = store_total["销售额"] / total
print(store_total)

   门店编号          销售额        占比
0  CDLG  10493.36542  0.379661
2  CDXL   9584.16710  0.346765
1  CDNL   7561.27355  0.273575


### 3.3一天中什么时段客流最大？如何优化排班和促销？

#### 2017年数据是同一天的数据，适合做同纬度分析，从data2017里面提取小时，统计每个小时的订单数

In [18]:
# 确保“成交时间”是 datetime 类型（如果尚未转换）
data2017['成交时间'] = pd.to_datetime(data2017['成交时间'])

# 提取小时
data2017['小时'] = data2017['成交时间'].dt.hour

# 按小时统计订单数（客流）
hourly_orders = data2017.groupby('小时').size().reset_index(name='订单数')

# 按小时统计销量（商品件数）
hourly_quantity = data2017.groupby('小时')['销量'].sum().reset_index(name='总销量')

# 查看结果
print("各小时订单数：")
print(hourly_orders.sort_values('订单数', ascending=False))
print("\n各小时总销量：")
print(hourly_quantity.sort_values('总销量', ascending=False))

各小时订单数：
    小时  订单数
4   10  533
3    9  511
2    8  372
12  19  294
10  17  232
11  18  230
5   11  211
9   16  166
1    7  152
13  20  124
7   14  113
8   15   96
6   13   90
0    6   63
14  21   44

各小时总销量：
    小时      总销量
4   10  612.397
3    9  547.642
2    8  441.830
12  19  324.049
0    6  323.288
11  18  242.301
5   11  220.405
10  17  214.644
1    7  157.216
9   16  155.862
13  20  136.018
6   13  111.473
7   14  109.143
8   15   92.016
14  21   44.877


无论是从订单数还是销量方面来看，10点都是一个高频时段

1. 增加高峰时段人力配置
9:00‑10:00 安排最多的人手，包括收银、导购、理货、客服等。
8:00 和 17:00‑19:00 同样需要充足人力，确保顾客体验。
2. 清晨6点专项处理
6点订单少但销量大，可能是批发或批量订单，建议安排专门的物流或打包小组提前到岗，快速处理大宗订单，避免影响后续高峰。
3. 低谷时段灵活用工
14:00‑16:00、20:00后订单稀少，可安排员工轮休、培训或进行补货、清洁等后台工作。可采用兼职或小时工，在高峰时段补充，低谷时段减少。

## 四 生成可视化报表

In [22]:
# 查看这些变量的前几行，确保没有报错
print(type_stats.head())
print(store_total.head())
print(hourly_orders.head())
print(hourly_quantity.head())

       商品ID     总销量    平均单价
0  29989025   1.000   2.500
1  29989047   0.278  17.615
2  29989048   7.831   1.980
3  29989049  11.822   1.980
4  29989050   0.279   6.790
   门店编号          销售额        占比
0  CDLG  10493.36542  0.379661
2  CDXL   9584.16710  0.346765
1  CDNL   7561.27355  0.273575
   小时  订单数
0   6   63
1   7  152
2   8  372
3   9  511
4  10  533
   小时      总销量
0   6  323.288
1   7  157.216
2   8  441.830
3   9  547.642
4  10  612.397


### 4.1 计算关键指标

In [23]:
# 总销售额
total_sales = data2017['销售额'].sum()

# 总订单数（使用订单ID去重计数）
total_orders = data2017['订单ID'].nunique()

# 平均客单价 = 总销售额 / 总订单数
avg_basket = total_sales / total_orders

# 制作一个 DataFrame 存放这些指标
summary = pd.DataFrame({
    '指标': ['总销售额', '总订单数', '平均客单价'],
    '数值': [total_sales, total_orders, avg_basket]
})

In [24]:
top_products = type_stats.sort_values('总销量', ascending=False).head(10)

In [25]:
low_products = type_stats.sort_values('总销量').head(10)

In [26]:
#合并小时数据
hourly_analysis = hourly_orders.merge(hourly_quantity, on='小时')

In [28]:
!pip install openpyxl



   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
with pd.ExcelWriter('超市销售报表_20170103.xlsx', engine='openpyxl') as writer:
    # 关键指标
    summary.to_excel(writer, sheet_name='关键指标', index=False)

    # 门店业绩
    store_total.to_excel(writer, sheet_name='门店业绩', index=False)

    # 热销商品
    top_products.to_excel(writer, sheet_name='热销商品', index=False)

    # 低销量商品
    low_products.to_excel(writer, sheet_name='低销量商品', index=False)

    # 客流时段
    hourly_analysis.to_excel(writer, sheet_name='客流时段', index=False)

    # 可选：保存原始清洗数据（供老板复核）
    data2017.to_excel(writer, sheet_name='原始数据_20170103', index=False)